# Domain 0.0 — Comprehensive Exercise

**Do this last.** Nothing new is taught here. Every cell assumes you have already worked the four domain folders, and the point is to find out which joins between them you cannot make yet.

## The problem this solves

Each domain notebook teaches one thing well, and each one hands you the setup you need before you start. Real work does not. A request arrives as "answer questions about these contracts and tell me what it cost" — one sentence that crosses document parsing, chunking, retrieval, access control and billing, and gives you none of the scaffolding. This notebook is that request.

## What you will be able to do

- Run one pipeline end to end, from files on a stage to a cost figure for the run
- Recognise which domain a failure belongs to, quickly, when a multi-step pipeline breaks
- Answer the five sample questions Snowflake publishes, and say why each distractor is tempting
- Find the gaps in your own recall before an exam finds them for you

## What this assumes you have already done

- **[Domain 1 — Gen AI Overview](../Domain%201.0%20-%20Gen%20AI%20Overview/)** — the Cortex capabilities, the two-part privilege rule, cross-region inference, embeddings and token counting
- **[Domain 2 — Gen AI Functions](../Domain%202.0%20-%20Gen%20AI%20Functions/)** — the AI function return shapes, chunking, Cortex Search, semantic views, bringing your own model
- **[Domain 3 — Gen AI Governance](../Domain%203.0%20-%20Gen%20AI%20Governance/)** — database roles, model access, the usage views, evaluation
- **[Domain 4 — Document Processing](../Domain%204.0%20-%20Document%20Processing/)** — `AI_PARSE_DOCUMENT`, `AI_EXTRACT`, staged files and their limits

Optionally the two deep dives, if containers and the registry are on your list: [DD1 — Snowpark Container Services](../Deep%20Dives/DD1%20-%20Snowpark%20Container%20Services.ipynb) and [DD2 — Model Registry](../Deep%20Dives/DD2%20-%20Model%20Registry.ipynb).

## Before you start

- `setup/dataset.sql` has been run, and the sample PDFs are staged to `@GENAI_STUDY.PUBLIC.DOCS_STAGE`.
- A role that can call AI functions: the `USE AI FUNCTIONS` account privilege plus one of `SNOWFLAKE.CORTEX_USER` / `SNOWFLAKE.AI_FUNCTIONS_USER`.
- `ACCOUNTADMIN` for the account-usage views at the end, or a role granted access to them.

📖 **Snowflake documentation for this notebook**
- [Cortex AISQL](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)
- [Cortex AI functions: privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
- [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)
- [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
- [METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)
- [CORTEX_AI_FUNCTIONS_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history)

---
## Part A — the five published sample questions

These are the questions printed in the official *SnowPro Specialty: Gen AI Exam Study Guide*. Work each one before reading the reasoning underneath it. What is worth your attention is not the right answer — it is why the wrong one looked right.

### Q1 — Daily AI cost

*A Gen AI Specialist needs to analyze the **daily** costs incurred for AI services in Snowflake. Which query retrieves credit consumption from Snowflake's metadata objects for data usage?*

- **A.** `SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY WHERE SERVICE_TYPE='AI_SERVICES';`
- **B.** `SELECT * FROM SNOWFLAKE.INFORMATION_SCHEMA.METERING_HISTORY WHERE SERVICE_TYPE='AI_SERVICES';`
- **C.** `SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY WHERE SERVICE_TYPE='AI_SERVICES';`
- **D.** `SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY WHERE SERVICE_TYPE='AI_SERVICES';`

**Answer: D.**

Two words decide it: **daily**, and the schema the view lives in.

| Option | Why it fails or wins |
|---|---|
| A | `QUERY_HISTORY` records query execution. It is not a metering view and does not carry `SERVICE_TYPE` credit columns |
| B | Wrong schema. The metering views used here are in `SNOWFLAKE.ACCOUNT_USAGE` |
| C | Right schema, right family — but `METERING_HISTORY` is **hourly**. It answers a slightly different question |
| D | ✅ `METERING_DAILY_HISTORY` is the daily-grain view, and `AI_SERVICES` is the documented `SERVICE_TYPE` value for Cortex AI usage |

C is the interesting distractor: nothing about it is wrong except the grain. When two options differ only by `_DAILY`, the qualifier in the stem *is* the question.

Its columns are `SERVICE_TYPE`, `USAGE_DATE`, `CREDITS_USED_COMPUTE`, `CREDITS_USED_CLOUD_SERVICES`, `CREDITS_USED`, `CREDITS_ADJUSTMENT_CLOUD_SERVICES` and `CREDITS_BILLED`. Latency runs up to 180 minutes, and 365 days are retained — so it is a trend tool, not a live meter.

→ [METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)

### Q2 — Memory in multi-turn chat

*What is the primary role of memory in a multi-turn chat conversation using a Gen AI model in Snowflake Cortex Analyst?*

- **A.** To securely store user credentials
- **B.** To increase the speed of response generation
- **C.** To maintain context throughout multiple requests
- **D.** To limit the number of tokens processed for each request

**Answer: C.**

The model is stateless. "Memory" is nothing more than the conversation history you resend on every call — the `messages` array of `system` / `user` / `assistant` turns documented on the legacy `SNOWFLAKE.CORTEX.COMPLETE` page, where one `system` turn is allowed and must come first. Its only job is to let turn *n* refer to what happened in turn *n−1*.

D is the appealing wrong answer because it is the **exact opposite** of what happens. Resending history makes each request *larger*, not smaller, and it grows every turn. That growth is why pruning and summarisation exist, and why `AI_COUNT_TOKENS` belongs in the loop rather than as an afterthought. A is unrelated. B is false for the same reason D is: more context is slower, not faster.

→ [Legacy COMPLETE and the messages array](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)

### Q3 — Two document types, two tools

*Invoices need specific fields extracted (vendor name, total amount, line items). Legal contracts need their full text converted to searchable Markdown with table structures preserved. How should the workflow be structured?*

- **A.** `AI_PARSE_DOCUMENT` for all, then SQL filters
- **B.** `AI_EXTRACT` for all, with separate response format schemas
- **C.** `AI_PARSE_DOCUMENT` in OCR mode for invoices, `AI_EXTRACT` for contracts
- **D.** `AI_EXTRACT` with a schema for invoices, `AI_PARSE_DOCUMENT` in LAYOUT mode for contracts

**Answer: D.**

Match the tool to the **shape of the output you need**, not to the type of document.

| Requirement | Right tool |
|---|---|
| named fields, as columns | `AI_EXTRACT` with a `responseFormat` |
| full text as Markdown, with tables intact | `AI_PARSE_DOCUMENT` with `{'mode': 'LAYOUT'}` |

| Option | Why it fails |
|---|---|
| A | SQL filters cannot reliably pull a vendor name out of prose. That is what the extraction is for |
| B | `AI_EXTRACT` returns the fields you asked for, not the whole document. It cannot produce full Markdown |
| C | Backwards on both halves — and OCR mode extracts text only, discarding the table structure the contracts need |
| D | ✅ Extraction where fields are wanted, LAYOUT parsing where structured full text is wanted |

Two details worth carrying: `AI_EXTRACT` reads the FILE directly, so the invoice path needs no parse step at all; and OCR is `AI_PARSE_DOCUMENT`'s default mode, which is why "just parse everything" quietly loses tables.

→ [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract) · [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)

### Q4 — Chatbot over PDFs

*An organization needs a chatbot that answers customer queries by searching through unstructured documents stored as PDFs. Which Snowflake capability enables building this chatbot?*

- **A.** Cortex Search
- **B.** Cortex Analyst
- **C.** Cortex Agent
- **D.** Snowflake CoWork (the feature Snowflake previously called Snowflake Intelligence; the exam study guide may still use the older name)

**Answer: A.**

Three of the four are plausible, so read the stem for what it actually names: **unstructured documents**, and **searching** them.

| Option | Its job |
|---|---|
| **A. Cortex Search** | ✅ retrieval over unstructured text — the capability that enables the chatbot |
| B. Cortex Analyst | text-to-SQL over **structured** tables via a semantic view. Wrong data type |
| C. Cortex Agent | an orchestrator, which would *call* Cortex Search. Not the retrieval capability |
| D. Snowflake CoWork | a ready-made application built on agents — now documented as Snowflake CoWork. A product, not the enabling capability |

The distinction being drawn is layering: Search is the **capability**, an Agent is the **orchestration** around it, and Intelligence / CoWork is the **application** on top. When a question asks which capability *enables* the use case, name the one doing the retrieval.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

### Q5 — Generating a lesson plan

*Which Snowflake Cortex LLM function should be used to generate an instructional lesson plan based on a prompt?*

- **A.** `AI_COMPLETE`
- **B.** `AI_EXTRACT`
- **C.** `SUMMARIZE`
- **D.** `AI_TRANSLATE`

**Answer: A.**

`AI_COMPLETE` is the only general-purpose generation function in the set. The other three are task-specific and all *reduce or transform* text that already exists rather than producing new content:

- `AI_EXTRACT` pulls named fields **out of** a document
- `SUMMARIZE` (`SNOWFLAKE.CORTEX.SUMMARIZE`) **shortens** existing text
- `AI_TRANSLATE` **converts** text between languages, with `''` as the source code meaning auto-detect

Note the guide writes `SUMMARIZE`, not `AI_SUMMARIZE`. The AISQL function list has no `AI_SUMMARIZE`; the aggregate `AI_SUMMARIZE_AGG` is a different function that summarises across rows.

**The rule underneath:** anything that produces new prose from an instruction is `AI_COMPLETE`. Reach for a task-specific function only when the output is a fixed shape — a label (`AI_CLASSIFY`), a boolean (`AI_FILTER`), named fields (`AI_EXTRACT`), a vector (`AI_EMBED`).

→ [Cortex AISQL](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)

> ### ⚠️ Common misconceptions
>
> **"A wrong answer on a cost question means I picked the wrong view family."**
> More often it means you picked the right family at the wrong **grain**. `METERING_HISTORY` is hourly and `METERING_DAILY_HISTORY` is daily; both are in `ACCOUNT_USAGE` and both filter on `SERVICE_TYPE`. Reading the stem for "daily", "hourly" or "per query" is the whole exercise.
> → [METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)
>
> **"Cortex Agents, Cortex Search and Snowflake CoWork are alternatives to each other."**
> They are layers. Search retrieves, an agent orchestrates tools including Search, and CoWork / Snowflake CoWork is a finished application built on agents. Treating them as competing options makes every "which one" question a coin flip; treating them as a stack makes the stem's wording decisive.
> → [Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)
>
> **"Adding conversation history is a way to keep requests small and fast."**
> It does the reverse. History is resent in full on every turn, so the request grows and the cost and latency grow with it. That is the reason multi-turn designs need pruning, summarisation and a token gate — not an optimisation, a containment measure.
> → [Legacy COMPLETE and the messages array](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)

---
## Part B — one pipeline, all four domains

Nine cells, run in order, each producing one output you can check. The value is in noticing *which* domain a step belongs to, because that is how you will locate a failure in work nobody has pre-labelled for you.

```
Domain 4   documents on a stage      ->  AI_EXTRACT / AI_PARSE_DOCUMENT
Domain 2   chunk -> embed            ->  the retrieval substrate
Domain 2   enrich                    ->  classify, redact, sentiment, translate
Domain 3   who may run this          ->  grants and model access
Domain 3   what did it cost          ->  the usage views
```

Every step here is a step you have run before in its own notebook. What is new is that nothing tells you which one you are in.

→ [More on AI_EXTRACT and its limits](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)
→ [More on AI_PARSE_DOCUMENT modes](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
→ [More on chunking with SPLIT_TEXT_RECURSIVE_CHARACTER](https://docs.snowflake.com/en/sql-reference/functions/split_text_recursive_character-snowflake-cortex)
→ [More on attributing Cortex spend](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history)

In [ ]:
%%sql
-- Step 0 -- session context and prerequisites, clubbed.
-- The QUERY_TAG set here is what makes Step 7's per-run cost attribution possible:
-- CORTEX_AI_FUNCTIONS_USAGE_HISTORY has a QUERY_TAG column, and you cannot tag after the fact.
USE ROLE ACCOUNTADMIN;
USE DATABASE GENAI_STUDY;
USE SCHEMA PUBLIC;
USE WAREHOUSE COMPUTE_WH;

ALTER SESSION SET QUERY_TAG = 'c02_comprehensive_exercise';
ALTER TABLE GENAI_STUDY.PUBLIC.SUPPORT_TICKETS SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r ex_extract
-- Step 1 (Domain 4) -- named fields straight out of a staged document, no parse step.
-- AI_EXTRACT reads the FILE directly; responseFormat says what you want, in questions.
-- Limits worth remembering: under 100 MB, at most 125 pages, up to 100 entity questions.
SELECT
    RELATIVE_PATH,
    AI_EXTRACT(
        file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', RELATIVE_PATH),
        responseFormat => {'vendor':  'What company issued this document?',
                           'total':   'What is the total amount due, as a number?',
                           'doc_date':'What is the document date in YYYY-MM-DD format?'}
    ):response AS fields
FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE)
WHERE LOWER(RELATIVE_PATH) LIKE '%.pdf'
  AND SIZE < 104857600;

In [ ]:
%%sql -r ex_parse
-- Step 2 (Domain 4) -- the same documents as full Markdown, for indexing.
-- LAYOUT extracts structure including tables; OCR (the default) would extract text only
-- and quietly flatten every table into a run of numbers.
SELECT
    RELATIVE_PATH,
    AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', RELATIVE_PATH),
        {'mode': 'LAYOUT'},
        TRUE                      -- return_error_details
    ):value:content::VARCHAR AS markdown
FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE)
WHERE LOWER(RELATIVE_PATH) LIKE '%.pdf';

In [ ]:
%%sql -r ex_chunk_embed
-- Step 3 (Domain 2) -- chunk, then embed, with the token count beside each chunk so you
-- can see whether you are anywhere near the model's 512-token context window.
-- Note the units differ: chunk_size counts CHARACTERS, the model's limit is in TOKENS.
SELECT
    t.ticket_id,
    f.index          AS chunk_index,
    f.value::VARCHAR AS chunk_text,
    AI_COUNT_TOKENS('AI_EMBED', 'snowflake-arctic-embed-m-v1.5', f.value::VARCHAR) AS chunk_tokens,
    AI_EMBED('snowflake-arctic-embed-m-v1.5', f.value::VARCHAR)                    AS embedding
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS t,
     LATERAL FLATTEN(
         SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(t.ticket_text, 'none', 400, 50, ['\n\n', '. ', ' '])
     ) f;

In [ ]:
%%sql -r ex_enrich
-- Step 4 (Domain 2) -- enrich four ways in one pass. Each function has its own return
-- shape, and reading the shape correctly is most of the work here:
--   AI_REDACT    -> text
--   AI_CLASSIFY  -> {"labels": [...]}
--   AI_SENTIMENT -> {"categories": [{"name": ..., "sentiment": ...}]}  -- a LABEL, not a number
--   AI_TRANSLATE -> text; '' as the source language means auto-detect
SELECT
    ticket_id,
    AI_REDACT(ticket_text)                                                          AS safe_text,
    AI_CLASSIFY(ticket_text, ['billing','technical','shipping']):labels[0]::VARCHAR AS category,
    (SELECT c.value:sentiment::VARCHAR
       FROM LATERAL FLATTEN(AI_SENTIMENT(ticket_text):categories) c
      WHERE c.value:name::VARCHAR = 'overall')                                      AS sentiment,
    CASE WHEN language = 'en' THEN ticket_text
         ELSE AI_TRANSLATE(ticket_text, '', 'en') END                               AS english_text
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;

> ### 🤔 Stop and think
>
> - Step 3 re-chunks and re-embeds every ticket on every run. That is fine here and wrong in production. What would you change, and what new problem does the change introduce — stale vectors, a refresh schedule, a table nobody owns?
> - Step 4 does four AI calls per row in one pass. It is the shortest code and not the cheapest plan. When would you split it, and what would you measure to decide?
> - Steps 7 and 8 report cost after the fact, with hours of latency. If someone runs this over a million documents by accident, what actually stops them — and is that control a technical one or a social one?
> - `AI_REDACT` runs *after* the text has already been read by the classification and sentiment calls in the same SELECT. Does the ordering in that statement mean what you want it to mean?

In [ ]:
%%sql -r ex_grants
-- Step 5 (Domain 3) -- who can actually run any of this?
-- Everything above ran as ACCOUNTADMIN, which proves nothing about whether a
-- least-privileged role could do the same work. This is where you find out.
SHOW GRANTS TO ROLE IDENTIFIER(CURRENT_ROLE());

In [ ]:
%%sql -r ex_models
-- Step 6 (Domain 3) -- which models can this role reach, and from where?
-- Two filters act at once: your model grants, and CORTEX_ENABLED_CROSS_REGION.
-- Read in_region_availability and cross_region_availability together.
SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS;

In [ ]:
%%sql -r ex_cost
-- Step 7 (Domain 3) -- what did this exercise cost? Attribution, thanks to the tag set
-- in Step 0. This view also carries QUERY_ID, ROLE_NAMES and USER_NAME.
SELECT
    FUNCTION_NAME,
    MODEL_NAME,
    COUNT(DISTINCT QUERY_ID) AS calls,
    SUM(CREDITS)             AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE QUERY_TAG = 'c02_comprehensive_exercise'
GROUP BY 1, 2
ORDER BY credits DESC;

In [ ]:
%%sql -r ex_daily_cost
-- Step 8 (Domain 3) -- the daily roll-up, at the grain sample question 1 asks for.
-- No QUERY_TAG here: this view knows SERVICE_TYPE and a date, nothing about who spent it.
SELECT USAGE_DATE, SERVICE_TYPE, CREDITS_USED, CREDITS_BILLED
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE = 'AI_SERVICES'
  AND USAGE_DATE >= DATEADD('day', -7, CURRENT_DATE)
ORDER BY USAGE_DATE DESC;

> ### ⚠️ Common misconceptions
>
> **"`QUERY_TAG` will let me attribute cost in any usage view."**
> It is a column in `CORTEX_AI_FUNCTIONS_USAGE_HISTORY`, alongside `QUERY_ID`, `ROLE_NAMES`, `USER_NAME` and `CREDITS` — which is why tagging the session at step 0 makes step 7 work. It is not in `METERING_DAILY_HISTORY`, which knows only `SERVICE_TYPE` and a date. Attribution and totals come from different views, and neither substitutes for the other.
> → [CORTEX_AI_FUNCTIONS_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history)
>
> **"The pipeline ran, so the governance is fine."**
> It ran as `ACCOUNTADMIN`, which is the one role that proves nothing. A pipeline is governed when it runs as the least-privileged role that can do the work, and the way to find out what that is, is to try it. Step 5 exists so that you ask the question before production does.
> → [Cortex AI functions: privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

---
## Part C — readiness checklist

Tick these from memory. Anything you hesitate on, reread the section it comes from.

### Domain 1 — Gen AI Overview (19%)
- [ ] Name each Cortex capability and the problem it is *for*, and explain Search / Agent / CoWork as layers rather than alternatives
- [ ] The two-part access rule: `USE AI FUNCTIONS` (or `USE AI FUNCTION <name>`) **plus** one of `CORTEX_USER` / `AI_FUNCTIONS_USER`
- [ ] Which SNOWFLAKE database roles exist, and that only `CORTEX_USER` and `COPILOT_USER` are granted to `PUBLIC` by default
- [ ] The `CORTEX_ENABLED_CROSS_REGION` values, that combinations are allowed, and that the finest granularity is cloud plus geography — never a single physical region
- [ ] Cross-region data handling: private backbone within a cloud, mTLS across clouds, nothing stored at the processing region, credits billed in the requesting region, no egress charge
- [ ] Embedding models by dimension and context window, and that `VECTOR` tops out at 4,096 dimensions
- [ ] Snowflake is an MCP **server**; a CKE is a published Cortex Search Service
- [ ] `cortex --continue` (most recent) versus `-r` / `--resume <session_id>` (a specific one)

### Domain 2 — Gen AI Functions
- [ ] `AI_CLASSIFY` returns `{"labels": [...]}`; `AI_SENTIMENT` returns `{"categories": [{name, sentiment}]}` with `positive` / `negative` / `neutral` / `mixed` / `unknown`
- [ ] `AI_EXTRACT` returns `{response, scoring, error}` — read `:response:<field>`
- [ ] `AI_TRANSLATE` auto-detects when the source language is `''`; `AI_SIMILARITY` takes its model in a config object and returns a float from −1 to 1
- [ ] `AI_COUNT_TOKENS`' first argument is the **function name**, and it estimates **input** tokens
- [ ] `SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(text, **format**, chunk_size, [overlap], [separators])` — `chunk_size` is characters, the model's limit is tokens
- [ ] Cortex Search: owner's rights, ≤ 512-token chunks, change tracking on base objects, `USAGE` on the service and its database and schema to query
- [ ] Model Registry: `target_platforms` decides the substrate, 15 GB for warehouse serving, `MODEL(<name>[, <version>])!<method>(...)` to call it

### Domain 3 — Gen AI Governance
- [ ] `AI_FUNCTIONS_USER` excludes the aggregates `AI_AGG` and `AI_SUMMARIZE_AGG`
- [ ] The narrow roles and what each one is for: `CORTEX_EMBED_USER`, `CORTEX_ANALYST_USER`, `CORTEX_AGENT_USER`, `CORTEX_REST_API_USER`
- [ ] `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` carries `QUERY_TAG`, `QUERY_ID`, `ROLE_NAMES`, `USER_NAME` and `CREDITS` — this is where attribution lives
- [ ] `SERVICE_TYPE = 'AI_SERVICES'` for Cortex AI functions and `'SNOWPARK_CONTAINER_SERVICES'` for SPCS, in `METERING_HISTORY` (hourly) and `METERING_DAILY_HISTORY` (daily)
- [ ] `MODEL_SERVING_USAGE_HISTORY` for registry serving, split by `INVOCATION_TYPE` (`SPCS` or `WAREHOUSE`)
- [ ] The five AI Observability evaluation metrics — coherence, answer relevance, groundedness, context relevance, correctness — and that correctness is the one compared against a recorded ground-truth output
- [ ] Disabling a feature means revoking its database role from `PUBLIC`, for example `REVOKE DATABASE ROLE SNOWFLAKE.COPILOT_USER FROM ROLE PUBLIC;`

### Domain 4 — Document Processing
- [ ] `AI_EXTRACT`: files under 100 MB, at most 125 pages, up to 100 entity questions or 10 table questions per call, 512-token entity output and 4096-token table output
- [ ] `AI_PARSE_DOCUMENT`: OCR is the default and extracts text only; LAYOUT extracts structure including tables, and can extract images
- [ ] `page_split` splits the document and processes each page separately, returning `{"pages": [{content, index}]}` with a 0-based index — PDF, PPTX and DOCX only
- [ ] `AI_EXTRACT(model => …)` is a **named** argument, used to point at a fine-tuned extraction model
- [ ] `return_error_details` on both functions, instead of wrapping the call in something that swallows the failure

---
### Final sanity check

If you can explain, without notes, **why you cannot write `AI_SENTIMENT(ticket_text):score > 0.7`** — that `AI_SENTIMENT` returns `{"categories": [{"name": …, "sentiment": …}]}`, where `sentiment` is a label string and not a number, so there is nothing there to compare with `>` — then you have the habit the whole exam rewards: knowing the *shape* of what comes back, not just the name of the function that returns it.

---

## Check your understanding

Twelve cross-domain questions. Each one needs at least two domains to answer. Work them before expanding.

**1.** A pipeline parses contracts with `AI_PARSE_DOCUMENT`, chunks them, and indexes the chunks in a Cortex Search service. HR asks why an analyst retrieved a document their row access policy should have hidden. Where did it go wrong, and at which step would you fix it?

<details><summary>Show answer</summary>

At indexing, not at query time. A Cortex Search service searches with **owner's rights**: the content was read once, by the owner, when the index was built, and the querying role's restrictions are not re-applied to it. There is no error and no empty result — just a hit the analyst should not have seen. The fix belongs at the Domain 4 / Domain 2 boundary: filter before you index, and build separate services per audience when the boundaries differ. Adding a `WHERE` clause to the query does not help, because the index has already flattened the distinction.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**2.** A 300-page contract must become searchable. `AI_EXTRACT` refuses it. What is the constraint, and what is the route through?

<details><summary>Show answer</summary>

`AI_EXTRACT` accepts at most **125 pages** (and files under 100 MB). The route is `AI_PARSE_DOCUMENT` with `page_split`, which splits the document and processes each page separately, returning `{"pages": [{content, index}]}` with a 0-based index — supported for PDF, PPTX and DOCX. From there it is a Domain 2 problem: chunk the per-page content to ≤ 512 tokens and index it. If you also need named fields, run `AI_EXTRACT` against the pages that carry them rather than the whole file.

→ [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract) · [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)

</details>

**3.** Finance wants per-team attribution for Cortex spend. You point them at `METERING_DAILY_HISTORY` and they say it is useless. Why, and what do you give them instead?

<details><summary>Show answer</summary>

`METERING_DAILY_HISTORY` has `SERVICE_TYPE`, `USAGE_DATE` and credit columns — a total, with no notion of who spent it. Attribution lives in `CORTEX_AI_FUNCTIONS_USAGE_HISTORY`, which carries `QUERY_TAG`, `QUERY_ID`, `ROLE_NAMES`, `USER_NAME`, `FUNCTION_NAME`, `MODEL_NAME` and `CREDITS`. The Domain 3 habit that makes it work is set in Domain 1: tag the session (`ALTER SESSION SET QUERY_TAG = …`) *before* the work runs, because you cannot tag it afterwards. Give them both — one for the total that reconciles with the bill, one for the breakdown.

→ [CORTEX_AI_FUNCTIONS_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history)

</details>

**4.** An application account should call the Cortex REST API to embed text, and nothing else. What do you grant, and what does each near-miss give away?

<details><summary>Show answer</summary>

`SNOWFLAKE.CORTEX_REST_API_USER` grants REST access without any other Cortex feature — and if it must also embed, `SNOWFLAKE.CORTEX_EMBED_USER` covers `AI_EMBED`, `EMBED_TEXT_768` and `EMBED_TEXT_1024`. The near-misses: `CORTEX_USER` works and hands the application every Cortex capability in the account; `AI_FUNCTIONS_USER` gives it every scalar AI function in SQL, which is more surface than "call one REST endpoint" asked for. Remember the account privilege too — `USE AI FUNCTIONS`, or a per-function `USE AI FUNCTION AI_EMBED`, is still required alongside whichever database role you pick.

→ [Snowflake database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)

</details>

**5.** A RAG pipeline embeds with `snowflake-arctic-embed-m-v1.5` into `VECTOR(FLOAT, 768)`. Parsed contract pages are producing empty or unhelpful retrieval. Name two plausible causes from different domains.

<details><summary>Show answer</summary>

**Domain 4:** the pages were parsed in OCR mode, which extracts text only, so the tables that carried the answer arrived as an unstructured run of numbers. LAYOUT mode preserves the structure. **Domain 2:** the chunks are far larger than the model's 512-token context window, so either the call is failing on length or a single vector is averaging away the specific clause the searcher wanted. Gate with `AI_COUNT_TOKENS('AI_EMBED', …)` and chunk with `SPLIT_TEXT_RECURSIVE_CHARACTER`, remembering that its `chunk_size` is characters while the limit is tokens. Both causes produce plausible-looking results rather than errors, which is what makes them expensive.

→ [Text embedding models](https://docs.snowflake.com/en/user-guide/snowflake-cortex/vector-embeddings)

</details>

**6.** Your account is `DISABLED` for cross-region inference. `SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS` returns fewer models than a colleague sees in another account. Is this a privilege problem?

<details><summary>Show answer</summary>

Not necessarily — it is two different filters at once. The command's output is filtered by the model grants your role holds (Domain 3), *and* what you can actually reach depends on `CORTEX_ENABLED_CROSS_REGION` (Domain 1), whose columns `in_region_availability` and `cross_region_availability` are in that same output. A third possibility is scope: without `IN SCHEMA SNOWFLAKE.MODELS` the command can return zero rows entirely. Check the parameter with `SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT` before assuming a grant is missing.

</details>

**7.** A model is logged to the registry for `WAREHOUSE` only. A product team now needs a 40 ms HTTP endpoint. What has to happen, and what does it cost?

<details><summary>Show answer</summary>

`target_platforms` is set at `log_model` time, so the version has to be logged again with `SNOWPARK_CONTAINER_SERVICES` (or both), then deployed with `create_service` — which needs `snowflake-ml-python` 1.25.0 or later, a compute pool, and `BIND SERVICE ENDPOINT` on the account if the endpoint is public. The cost is Domain 3's problem, not Domain 2's: a compute pool bills node-hours in `IDLE` as well as `ACTIVE`, so the endpoint bills whether or not the product team calls it. Set the service's own `AUTO_SUSPEND_SECS` — it defaults to 0, meaning off — if there are quiet hours.

→ [Model serving in SPCS](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/real-time-inference-rest-api)

</details>

**8.** Your nightly enrichment job worked for months, then started failing on one step only. The role was recently changed from `CORTEX_USER` to `AI_FUNCTIONS_USER`. Which step?

<details><summary>Show answer</summary>

Whichever one calls `AI_AGG` or `AI_SUMMARIZE_AGG`. `AI_FUNCTIONS_USER` covers the scalar AI functions and deliberately excludes the two aggregates, so `AI_CLASSIFY`, `AI_SENTIMENT`, `AI_TRANSLATE` and `AI_EMBED` keep working while the per-group summary step stops. The diagnostic instinct worth building: when *one* step of a working pipeline fails after a role change, look for the function whose privilege boundary differs from its neighbours', not for something wrong with the data.

→ [Cortex AI functions: privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**9.** You are asked to expose a document-question-answering capability to an external agent framework. Sketch the design across domains, and name what you must already hold.

<details><summary>Show answer</summary>

Domain 4 gets the text out — `AI_PARSE_DOCUMENT` in LAYOUT mode over the staged files. Domain 2 chunks it to ≤ 512 tokens and builds a Cortex Search service, which needs `CORTEX_USER` or `CORTEX_EMBED_USER`, `CREATE CORTEX SEARCH SERVICE` on the schema, `SELECT` on the base tables, `USAGE` on the refresh warehouse, and **change tracking enabled** on the underlying objects. Domain 1 publishes it: `CREATE MCP SERVER` with a `CORTEX_SEARCH_SERVICE_QUERY` tool, since Snowflake is the MCP **server** here. You need `CREATE MCP SERVER` and `USAGE` on the schema plus `USAGE` on the search service — exposing a tool never grants you access to what is behind it. Domain 3 is the part people skip: the service searches with owner's rights, so what you index is what every external caller can retrieve.

→ [CREATE MCP SERVER](https://docs.snowflake.com/en/sql-reference/sql/create-mcp-server)

</details>

**10.** Two designs for the same RAG application: index everything into one Cortex Search service with a filter at query time, or build one service per audience. Argue both sides.

<details><summary>Show answer</summary>

**One service:** a single pipeline, one refresh schedule, one object to grant and monitor, and a smaller bill. It relies on a filter to enforce a boundary that the index itself does not enforce — and because services run with owner's rights, a bug in that filter is a silent disclosure rather than an error. **Per audience:** the boundary is structural, so there is nothing to get wrong at query time; you pay in duplicated content, duplicated indexing cost, and n pipelines to keep in step. The deciding question is not cost but consequence: if a leak between two audiences is a compliance incident, the structural answer is worth the duplication; if the audiences differ only by convenience, it is not.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**11.** Your team must choose between fine-tuning `llama3.1-8b` on 5,000 labelled examples and building a retrieval pipeline over the same corpus. Frame the decision honestly.

<details><summary>Show answer</summary>

Ask what is wrong with the current answers. Wrong *behaviour* — register, format, house vocabulary — is what fine-tuning fixes, and 5,000 `prompt` / `completion` pairs is a workable dataset, with a 24k-token context window (20k prompt, 4k completion) and `CREATE MODEL` on the target schema required. Wrong *facts* is what retrieval fixes, because a tuned model still has a fixed cut-off and every refresh means another tuning run, another version to govern and another round of credits. The costs are not symmetric: fine-tuning front-loads effort and creates an asset that decays; retrieval creates a pipeline somebody maintains forever but keeps the content editable this afternoon. Most teams should try prompting, then retrieval, then tuning.

→ [Cortex Fine-tuning](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)

</details>

**12.** Leadership asks whether the RAG assistant is "good enough to ship". What do you measure, and what can measurement not tell you?

<details><summary>Show answer</summary>

AI Observability computes coherence, answer relevance, groundedness, context relevance and correctness — and they split cleanly across the pipeline: **context relevance** grades retrieval (Domain 2, and upstream of it the Domain 4 parsing), **groundedness** grades whether the generated answer is actually supported by what was retrieved, and **answer relevance** grades whether it addressed the question. **Correctness** is the one compared against a recorded ground-truth output, which means somebody has to write that ground truth first. What the numbers cannot tell you is whether the corpus should have been indexed at all: a service running with owner's rights can score superbly on every metric while retrieving documents the asker was never entitled to see. Evaluation grades the answer, not the entitlement.

→ [Evaluate AI applications](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)

</details>